# NB13 — Robustness, Pathway Recovery & Supergroup Analysis

**Three jobs in one notebook:**

| Job | What it does | Why it matters |
|-----|-------------|----------------|
| **A — Multi-seed CI** | Re-runs OT pilot across 5 seeds, computes bootstrap 95% CI | Proves gains are not noise; required by reviewer |
| **B — Pathway recovery** | OT vs control baseline on Hallmark/KEGG enrichment (NB02-style) | Biological validity beyond R²; strongest figure |
| **C — Supergroup analysis** | Per-supergroup OT gain vs scaffold Tanimoto distance | Explains *why* OOD gain is small; mechanistic story |

**Prerequisites:** NB10 + NB11 artifacts must be on Drive. NB12 logic is reproduced here — no need to re-run NB12.

**Colab strategy:** Each seed run is checkpointed. If session dies, re-run from the last completed seed.

## 0) Drive mount + installs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install rdkit anndata scanpy scikit-learn scipy pandas numpy gseapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 90.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas

## 1) Config — single place to change all paths

In [3]:
import os, json, pickle, random, warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from scipy.linalg import sqrtm

warnings.filterwarnings('ignore')

@dataclass
class CFG:
    # ── paths (match your NB10/NB11 exactly) ──────────────────────────
    data_path:     str = '/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad'
    nb10_res_dir:  str = '/content/drive/MyDrive/ChemDFM/results_nb10_scaffold'
    nb10_art_dir:  str = '/content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts'
    nb11_res_dir:  str = '/content/drive/MyDrive/ChemDFM/results_nb11_scaffold'
    results_dir:   str = '/content/drive/MyDrive/ChemDFM/results_nb13'

    # ── column names ──────────────────────────────────────────────────
    condition_col: str = 'condition'
    cell_col:      str = 'cell_type'
    dose_col:      str = 'dose'
    fallback_dose: str = 'dose_val'
    control_label: str = 'control'
    split_col:     str = 'split_scaffold'

    # ── model dims (must match NB11) ──────────────────────────────────
    pca_dim:       int = 25

    # ── OT config ─────────────────────────────────────────────────────
    n_supergroups:        int   = 8
    cov_reg:              float = 1e-3
    min_cells_per_group:  int   = 10

    # ── multi-seed config ─────────────────────────────────────────────
    seeds:         tuple = (42, 7, 13, 99, 2025)
    n_bootstrap:   int   = 1000   # bootstrap resamples for CI

    # ── pathway config ────────────────────────────────────────────────
    top_k_genes:   int   = 100    # top DEGs for enrichment
    gene_sets:     tuple = ('MSigDB_Hallmark_2020',)  # gseapy set name

cfg = CFG()
Path(cfg.results_dir).mkdir(parents=True, exist_ok=True)
print('Config ready. Results ->', cfg.results_dir)

Config ready. Results -> /content/drive/MyDrive/ChemDFM/results_nb13


## 2) Load NB10/NB11 artifacts

In [4]:
def _pkl(path, name):
    if not os.path.exists(path):
        raise FileNotFoundError(f'FATAL: {name} not found at {path}')
    with open(path, 'rb') as f:
        return pickle.load(f)

pca        = _pkl(f'{cfg.nb11_res_dir}/pca_model_nb11.pkl',        'PCA')
ctrl_means = _pkl(f'{cfg.nb11_res_dir}/ctrl_means_pca_nb11.pkl',   'ctrl_means_pca')
drug_enc   = _pkl(f'{cfg.nb11_res_dir}/drug_encoder_nb11.pkl',     'drug_encoder')
cell_enc   = _pkl(f'{cfg.nb11_res_dir}/cell_encoder_nb11.pkl',     'cell_encoder')

# ctrl_means_gene for pathway recovery (gene space)
ctrl_means_gene_path = f'{cfg.nb11_res_dir}/ctrl_means_gene_nb11.pkl'
ctrl_means_gene = _pkl(ctrl_means_gene_path, 'ctrl_means_gene') if os.path.exists(ctrl_means_gene_path) else None

drug_df = pd.read_csv(f'{cfg.nb10_res_dir}/scaffold_split_map.csv')
print(f'Loaded artifacts. PCA dim={pca.n_components}, drugs={len(drug_enc.classes_)}')
print(f'ctrl_means_gene available: {ctrl_means_gene is not None}')

Loaded artifacts. PCA dim=25, drugs=188
ctrl_means_gene available: True


## 3) Load data + project to PCA

In [5]:
adata = ad.read_h5ad(cfg.data_path)

# dose fallback
if cfg.dose_col not in adata.obs.columns and cfg.fallback_dose in adata.obs.columns:
    adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose]

# attach scaffold split from NB10
obs_csv = f'{cfg.nb10_art_dir}/obs_with_scaffold_split.csv'
obs_nb10 = pd.read_csv(obs_csv, index_col=0)
common_idx = adata.obs_names.intersection(obs_nb10.index)
adata = adata[common_idx].copy()
adata.obs[cfg.split_col] = obs_nb10.loc[adata.obs_names, cfg.split_col].values

# raw gene expression (for pathway recovery)
X_raw = adata.X
if hasattr(X_raw, 'toarray'):
    X_raw = X_raw.toarray()
X_raw = np.asarray(X_raw, dtype=np.float32)

# PCA projection (NB11 fit — no refit)
X_pca = pca.transform(X_raw).astype(np.float32)

# masks
split_vals = adata.obs[cfg.split_col].values
cond_vals  = adata.obs[cfg.condition_col].astype(str).values
cell_vals  = adata.obs[cfg.cell_col].astype(str).values

is_train   = (split_vals == 'train')   & (cond_vals != cfg.control_label)
is_test    = (split_vals == 'test')    & (cond_vals != cfg.control_label)
is_ood     = (split_vals == 'ood')     & (cond_vals != cfg.control_label)
is_control = (split_vals == 'control')

drug_to_scaffold = drug_df.set_index('drug')['scaffold'].to_dict()
drug_to_split    = drug_df.set_index('drug')['split_scaffold'].to_dict() if 'split_scaffold' in drug_df.columns else {}

cell_types = sorted(adata.obs[cfg.cell_col].astype(str).unique())
gene_names = list(adata.var_names)

print(f'Data: {adata.n_obs:,} cells, {adata.n_vars:,} genes')
print(f'Train: {is_train.sum():,} | Test: {is_test.sum():,} | OOD: {is_ood.sum():,}')

Data: 354,640 cells, 2,000 genes
Train: 240,527 | Test: 51,243 | OOD: 49,866


## 4) Core OT functions

Reproduced from NB12 — no dependency on NB12 pickle files.

In [6]:
def fit_gaussian_ot_map(X_src, X_tgt, reg=1e-3):
    dim = X_src.shape[1]
    mu_s = X_src.mean(0).astype(np.float64)
    mu_t = X_tgt.mean(0).astype(np.float64)
    Sig_s = np.cov(X_src.T) + reg * np.eye(dim)
    Sig_t = np.cov(X_tgt.T) + reg * np.eye(dim)
    S_half = sqrtm(Sig_s).real
    S_inv  = np.linalg.inv(S_half)
    M = sqrtm(S_half @ Sig_t @ S_half).real
    A = (S_inv @ M @ S_inv).astype(np.float32)
    return mu_s.astype(np.float32), mu_t.astype(np.float32), A

def apply_ot_map(x, mu_s, mu_t, A):
    return (mu_t + (A @ (x - mu_s).T).T).astype(np.float32)

def get_ctrl_anchor(ct):
    cm = ctrl_means[ct]
    return np.asarray(cm, dtype=np.float32)

def rowwise_pearson(a, b):
    vals = [pearsonr(a[i], b[i])[0]
            for i in range(a.shape[0])
            if np.std(a[i]) > 1e-8 and np.std(b[i]) > 1e-8]
    return float(np.mean(vals)) if vals else np.nan

def compute_metrics(pred, true, x0):
    out = {'r2_full': float(r2_score(true.ravel(), pred.ravel())),
           'pearson_row': rowwise_pearson(true, pred),
           'n_cells': int(pred.shape[0])}
    for k in [20, 50]:
        vals = []
        for i in range(true.shape[0]):
            top_idx = np.argsort(-np.abs(true[i] - x0[i]))[:k]
            if np.std(true[i, top_idx]) > 1e-8:
                vals.append(float(r2_score(true[i, top_idx], pred[i, top_idx])))
        out[f'r2_top{k}'] = float(np.mean(vals)) if vals else np.nan
    return out

print('OT functions ready.')

OT functions ready.


## 5) JOB A — Multi-seed OT evaluation with bootstrap CI

For each seed: re-shuffle supergroup KMeans init, re-fit OT maps, evaluate.
Then compute 95% CI via bootstrap resampling across seeds.

In [7]:
def run_ot_one_seed(seed):
    """Full OT pipeline for one random seed. Returns metrics dict."""
    rng = np.random.default_rng(seed)

    # ── drug centroids (train) ─────────────────────────────────────────
    train_drugs = sorted(set(cond_vals[is_train]))
    drug_centroids = {d: X_pca[is_train & (cond_vals == d)].mean(0)
                      for d in train_drugs
                      if (is_train & (cond_vals == d)).sum() > 0}

    # ── scaffold centroids ────────────────────────────────────────────
    scaffold_drugs = {}
    for drug, centroid in drug_centroids.items():
        scaf = drug_to_scaffold.get(drug, f'hash_{abs(hash(drug)) % 9999}')
        scaffold_drugs.setdefault(scaf, []).append(drug)

    scaffold_centroids = {
        scaf: np.mean([drug_centroids[d] for d in drugs], axis=0)
        for scaf, drugs in scaffold_drugs.items()
    }
    train_scaffolds_list = sorted(scaffold_centroids.keys())
    S = np.stack([scaffold_centroids[s] for s in train_scaffolds_list])

    # ── KMeans supergroups (seed-dependent) ──────────────────────────
    n_sg = min(cfg.n_supergroups, len(train_scaffolds_list))
    kmeans = KMeans(n_clusters=n_sg, random_state=int(seed), n_init=10)
    kmeans.fit(S)
    sg_labels = kmeans.labels_

    scaffold_to_sg = {scaf: int(sg_labels[i]) for i, scaf in enumerate(train_scaffolds_list)}
    drug_to_sg = {drug: scaffold_to_sg[drug_to_scaffold.get(drug, '?')]
                  for drug in drug_centroids
                  if drug_to_scaffold.get(drug, '?') in scaffold_to_sg}

    sg_centers = kmeans.cluster_centers_

    def route_drug(drug, split_mask_arr):
        mask = split_mask_arr & (cond_vals == drug)
        if mask.sum() == 0:
            return 0
        centroid = X_pca[mask].mean(0)
        return int(np.argmin(np.linalg.norm(sg_centers - centroid[None, :], axis=1)))

    all_drug_sg = dict(drug_to_sg)
    for split_mask_arr in [is_test, is_ood]:
        for drug in set(cond_vals[split_mask_arr]):
            if drug not in all_drug_sg:
                all_drug_sg[drug] = route_drug(drug, split_mask_arr)

    # ── fit OT maps ───────────────────────────────────────────────────
    ot_maps = {}
    for sg_id in range(n_sg):
        ot_maps[sg_id] = {}
        sg_drugs = [d for d, g in drug_to_sg.items() if g == sg_id]
        is_sg_train = is_train & np.isin(cond_vals, sg_drugs)
        for ct in cell_types:
            is_ct  = cell_vals == ct
            X_src  = X_pca[is_control & is_ct]
            X_tgt  = X_pca[is_sg_train & is_ct]
            if len(X_src) >= cfg.min_cells_per_group and len(X_tgt) >= cfg.min_cells_per_group:
                try:
                    mu_s, mu_t, A = fit_gaussian_ot_map(X_src, X_tgt, cfg.cov_reg)
                    ot_maps[sg_id][ct] = ('gaussian', mu_s, mu_t, A)
                except Exception:
                    X_src_m, X_tgt_m = X_src.mean(0).astype(np.float32), X_tgt.mean(0).astype(np.float32)
                    ot_maps[sg_id][ct] = ('mean_shift', X_src_m, X_tgt_m, None)
            else:
                ot_maps[sg_id][ct] = None

    # global fallback
    global_maps = {}
    for ct in cell_types:
        is_ct = cell_vals == ct
        X_src = X_pca[is_control & is_ct]
        X_tgt = X_pca[is_train & is_ct]
        if len(X_src) >= 2 and len(X_tgt) >= 2:
            try:
                global_maps[ct] = fit_gaussian_ot_map(X_src, X_tgt, cfg.cov_reg)
            except Exception:
                global_maps[ct] = (X_src.mean(0).astype(np.float32), X_tgt.mean(0).astype(np.float32), None)
        else:
            global_maps[ct] = (X_src.mean(0).astype(np.float32), X_tgt.mean(0).astype(np.float32), None)

    # ── predict ───────────────────────────────────────────────────────
    def predict_split(split_mask_arr):
        idxs = np.where(split_mask_arr)[0]
        pred_ot   = np.zeros((len(idxs), cfg.pca_dim), dtype=np.float32)
        pred_ctrl = np.zeros((len(idxs), cfg.pca_dim), dtype=np.float32)
        true_pca  = X_pca[idxs]
        for out_i, idx in enumerate(idxs):
            drug, ct = cond_vals[idx], cell_vals[idx]
            x0    = get_ctrl_anchor(ct)
            sg_id = all_drug_sg.get(drug, 0)
            entry = ot_maps.get(sg_id, {}).get(ct)
            if entry is None:
                mu_s, mu_t, A = global_maps[ct]
            else:
                _, mu_s, mu_t, A = entry
            x_pred = apply_ot_map(x0.reshape(1,-1), mu_s, mu_t, A).ravel() if A is not None else x0 + (mu_t - mu_s)
            pred_ot[out_i]   = x_pred
            pred_ctrl[out_i] = x0
        return pred_ot, pred_ctrl, true_pca

    results = {}
    for split_name, split_mask_arr in [('test', is_test), ('ood', is_ood)]:
        pred_ot, pred_ctrl, true_pca = predict_split(split_mask_arr)
        cv = cell_vals[split_mask_arr]
        results[f'ot_{split_name}']   = compute_metrics(pred_ot,   true_pca, pred_ctrl)
        results[f'ctrl_{split_name}'] = compute_metrics(pred_ctrl, true_pca, pred_ctrl)
        # per cell type
        for ct in cell_types:
            m = (cv == ct)
            if m.sum() == 0:
                continue
            results[f'ot_{split_name}_{ct}']   = compute_metrics(pred_ot[m],   true_pca[m], pred_ctrl[m])
            results[f'ctrl_{split_name}_{ct}'] = compute_metrics(pred_ctrl[m], true_pca[m], pred_ctrl[m])
        # store predictions for supergroup analysis (only seed=42)
        if seed == 42:
            results[f'_pred_ot_{split_name}']   = pred_ot
            results[f'_pred_ctrl_{split_name}'] = pred_ctrl
            results[f'_true_{split_name}']      = true_pca
            results[f'_idxs_{split_name}']      = np.where(split_mask_arr)[0]
            results['_all_drug_sg'] = all_drug_sg
            results['_drug_to_sg']  = drug_to_sg

    return results


# ── Run all seeds (checkpoint each) ──────────────────────────────────────
all_seed_results = {}
for seed in cfg.seeds:
    ckpt_path = f"{cfg.results_dir}/seed_{seed}_results.pkl"
    if os.path.exists(ckpt_path):
        print(f'Seed {seed}: loading from checkpoint...')
        with open(ckpt_path, 'rb') as f:
            all_seed_results[seed] = pickle.load(f)
    else:
        print(f'Seed {seed}: running...')
        result = run_ot_one_seed(seed)
        with open(ckpt_path, 'wb') as f:
            pickle.dump(result, f)
        all_seed_results[seed] = result
        print(f'  test r2_top50={result["ot_test"]["r2_top50"]:.4f}  '
              f'ood r2_top50={result["ot_ood"]["r2_top50"]:.4f}')

print('\nAll seeds done.')

Seed 42: running...
  test r2_top50=0.5628  ood r2_top50=0.5599
Seed 7: running...
  test r2_top50=0.5628  ood r2_top50=0.5598
Seed 13: running...
  test r2_top50=0.5628  ood r2_top50=0.5599
Seed 99: running...
  test r2_top50=0.5626  ood r2_top50=0.5598
Seed 2025: running...
  test r2_top50=0.5629  ood r2_top50=0.5606

All seeds done.


In [8]:
# ── Bootstrap 95% CI ──────────────────────────────────────────────────────
metrics_to_report = ['r2_top50', 'r2_top20', 'pearson_row']
splits_to_report  = ['test', 'ood']

def bootstrap_ci(values, n=1000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    boots = [np.mean(rng.choice(values, size=len(values), replace=True)) for _ in range(n)]
    lo = np.percentile(boots, 100 * alpha / 2)
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return float(np.mean(values)), lo, hi

ci_rows = []
seed_list = list(all_seed_results.keys())

for split in splits_to_report:
    for method_prefix in ['ot', 'ctrl']:
        key = f'{method_prefix}_{split}'
        for metric in metrics_to_report:
            vals = [all_seed_results[s][key][metric]
                    for s in seed_list
                    if key in all_seed_results[s] and not np.isnan(all_seed_results[s][key][metric])]
            if not vals:
                continue
            mean, lo, hi = bootstrap_ci(np.array(vals), n=cfg.n_bootstrap)
            ci_rows.append({
                'split': split, 'method': method_prefix,
                'metric': metric,
                'mean': round(mean, 4),
                'ci_lo': round(lo, 4),
                'ci_hi': round(hi, 4),
                'n_seeds': len(vals),
                'values_per_seed': vals
            })

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(f'{cfg.results_dir}/ci_table_nb13.csv', index=False)

# ── Print summary table ───────────────────────────────────────────────────
print('=' * 70)
print('MULTI-SEED BOOTSTRAP 95% CI  (5 seeds, 1000 resamples)')
print('=' * 70)
for split in splits_to_report:
    print(f'\n  [{split.upper()}]')
    for metric in metrics_to_report:
        row_ot   = ci_df[(ci_df.split==split) & (ci_df.method=='ot')   & (ci_df.metric==metric)]
        row_ctrl = ci_df[(ci_df.split==split) & (ci_df.method=='ctrl') & (ci_df.metric==metric)]
        if row_ot.empty or row_ctrl.empty:
            continue
        ot_m, ot_lo, ot_hi     = row_ot.iloc[0][['mean','ci_lo','ci_hi']]
        ct_m, ct_lo, ct_hi     = row_ctrl.iloc[0][['mean','ci_lo','ci_hi']]
        gain = ot_m - ct_m
        ci_overlap = (ot_lo < ct_hi)  # True = CIs overlap = gain not significant
        sig_str = '(CIs OVERLAP — not significant)' if ci_overlap else '*** SIGNIFICANT ***'
        print(f'    {metric:15s}  OT={ot_m:.4f} [{ot_lo:.4f},{ot_hi:.4f}]  '
              f'ctrl={ct_m:.4f} [{ct_lo:.4f},{ct_hi:.4f}]  '
              f'Δ={gain:+.4f}  {sig_str}')

print(f'\nSaved: ci_table_nb13.csv')

MULTI-SEED BOOTSTRAP 95% CI  (5 seeds, 1000 resamples)

  [TEST]
    r2_top50         OT=0.5628 [0.5627,0.5629]  ctrl=0.5291 [0.5291,0.5291]  Δ=+0.0337  *** SIGNIFICANT ***
    r2_top20         OT=0.4835 [0.4833,0.4836]  ctrl=0.4475 [0.4475,0.4475]  Δ=+0.0360  *** SIGNIFICANT ***
    pearson_row      OT=0.7790 [0.7789,0.7790]  ctrl=0.7507 [0.7507,0.7507]  Δ=+0.0283  *** SIGNIFICANT ***

  [OOD]
    r2_top50         OT=0.5600 [0.5598,0.5603]  ctrl=0.5500 [0.5500,0.5500]  Δ=+0.0100  *** SIGNIFICANT ***
    r2_top20         OT=0.4787 [0.4784,0.4791]  ctrl=0.4676 [0.4676,0.4676]  Δ=+0.0111  *** SIGNIFICANT ***
    pearson_row      OT=0.7741 [0.7740,0.7742]  ctrl=0.7680 [0.7680,0.7680]  Δ=+0.0061  *** SIGNIFICANT ***

Saved: ci_table_nb13.csv


In [9]:
# ── Per-cell-type CI ──────────────────────────────────────────────────────
cell_ci_rows = []
for ct in cell_types:
    for split in ['test', 'ood']:
        for method in ['ot', 'ctrl']:
            key = f'{method}_{split}_{ct}'
            vals = [all_seed_results[s][key]['r2_top50']
                    for s in seed_list
                    if key in all_seed_results[s] and not np.isnan(all_seed_results[s][key]['r2_top50'])]
            if not vals: continue
            mean, lo, hi = bootstrap_ci(np.array(vals), n=cfg.n_bootstrap)
            cell_ci_rows.append({'cell_type': ct, 'split': split, 'method': method,
                                  'r2_top50_mean': round(mean,4),
                                  'ci_lo': round(lo,4), 'ci_hi': round(hi,4)})

cell_ci_df = pd.DataFrame(cell_ci_rows)
cell_ci_df.to_csv(f'{cfg.results_dir}/ci_per_cell_nb13.csv', index=False)

print('Per-cell-type CI (r2_top50):')
print(cell_ci_df.to_string(index=False))

Per-cell-type CI (r2_top50):
cell_type split method  r2_top50_mean  ci_lo  ci_hi
     A549  test     ot         0.7077 0.7077 0.7078
     A549  test   ctrl         0.6791 0.6791 0.6791
     A549   ood     ot         0.7039 0.7037 0.7041
     A549   ood   ctrl         0.6956 0.6956 0.6956
     K562  test     ot         0.5835 0.5829 0.5841
     K562  test   ctrl         0.5617 0.5617 0.5617
     K562   ood     ot         0.5772 0.5767 0.5777
     K562   ood   ctrl         0.5697 0.5697 0.5697
     MCF7  test     ot         0.4827 0.4825 0.4829
     MCF7  test   ctrl         0.4410 0.4410 0.4410
     MCF7   ood     ot         0.4819 0.4816 0.4822
     MCF7   ood   ctrl         0.4698 0.4698 0.4698


## 6) JOB B — Pathway Recovery (NB02-style)

Top-K DEGs from OT predictions vs control baseline predictions.
Enrichment via gseapy Enrichr (Hallmark gene sets).
Metric: Jaccard overlap of enriched pathways, Pearson of enrichment scores.

In [10]:
try:
    import gseapy as gp
    GSEAPY_OK = True
    print('gseapy available.')
except ImportError:
    GSEAPY_OK = False
    print('gseapy not available — pathway recovery will be skipped.')

# ── PCA inverse transform → gene space ───────────────────────────────────
# Use seed=42 predictions stored earlier
seed42 = all_seed_results[42]

def pca_to_gene(X_pca_arr):
    """Inverse transform PCA predictions back to gene space."""
    return pca.inverse_transform(X_pca_arr).astype(np.float32)

if GSEAPY_OK and ctrl_means_gene is not None:

    def get_top_degs(pred_gene, ctrl_gene_arr, k=100):
        """Get top-k DEGs by mean absolute fold change."""
        delta = np.abs(pred_gene - ctrl_gene_arr).mean(axis=0)
        top_idx = np.argsort(-delta)[:k]
        return [gene_names[i] for i in top_idx]

    def run_enrichr(gene_list, gene_set='MSigDB_Hallmark_2020'):
        try:
            enr = gp.enrichr(gene_list=gene_list,
                             gene_sets=gene_set,
                             organism='Human',
                             outdir=None,
                             verbose=False)
            df = enr.results[['Term', 'Adjusted P-value', 'Combined Score']].copy()
            df = df[df['Adjusted P-value'] < 0.05].sort_values('Combined Score', ascending=False)
            return df
        except Exception as e:
            print(f'  Enrichr error: {e}')
            return pd.DataFrame(columns=['Term', 'Adjusted P-value', 'Combined Score'])

    pathway_rows = []
    for split_name in ['test', 'ood']:
        pred_ot_pca   = seed42[f'_pred_ot_{split_name}']
        pred_ctrl_pca = seed42[f'_pred_ctrl_{split_name}']
        true_pca_arr  = seed42[f'_true_{split_name}']
        idxs          = seed42[f'_idxs_{split_name}']
        cv_split      = cell_vals[idxs]

        pred_ot_gene   = pca_to_gene(pred_ot_pca)
        pred_ctrl_gene = pca_to_gene(pred_ctrl_pca)
        true_gene      = pca_to_gene(true_pca_arr)

        # Per cell type
        for ct in cell_types:
            ct_mask = (cv_split == ct)
            if ct_mask.sum() < 5:
                continue
            ctrl_gene_ct = ctrl_means_gene.get(ct)
            if ctrl_gene_ct is None:
                continue
            ctrl_gene_ct = np.asarray(ctrl_gene_ct, dtype=np.float32).reshape(1, -1)

            degs_ot   = get_top_degs(pred_ot_gene[ct_mask],   ctrl_gene_ct, k=cfg.top_k_genes)
            degs_ctrl = get_top_degs(pred_ctrl_gene[ct_mask], ctrl_gene_ct, k=cfg.top_k_genes)
            degs_true = get_top_degs(true_gene[ct_mask],      ctrl_gene_ct, k=cfg.top_k_genes)

            # Jaccard: predicted DEGs vs true DEGs
            jac_ot   = len(set(degs_ot)   & set(degs_true)) / len(set(degs_ot)   | set(degs_true))
            jac_ctrl = len(set(degs_ctrl) & set(degs_true)) / len(set(degs_ctrl) | set(degs_true))

            print(f'  [{split_name}] {ct}: DEG Jaccard  OT={jac_ot:.3f}  ctrl={jac_ctrl:.3f}  Δ={jac_ot-jac_ctrl:+.3f}')

            # Enrichr on true DEGs
            enr_true = run_enrichr(degs_true)
            enr_ot   = run_enrichr(degs_ot)
            enr_ctrl = run_enrichr(degs_ctrl)

            if len(enr_true) > 0:
                true_terms = set(enr_true['Term'])
                ot_terms   = set(enr_ot['Term'])   if len(enr_ot)   > 0 else set()
                ctrl_terms = set(enr_ctrl['Term']) if len(enr_ctrl) > 0 else set()
                recall_ot   = len(ot_terms   & true_terms) / len(true_terms) if true_terms else 0
                recall_ctrl = len(ctrl_terms & true_terms) / len(true_terms) if true_terms else 0
                print(f'          Pathway recall: OT={recall_ot:.3f}  ctrl={recall_ctrl:.3f}  '
                      f'n_true_pathways={len(true_terms)}')
            else:
                recall_ot = recall_ctrl = 0.0
                print(f'          No significant pathways in true DEGs for {ct}.')

            pathway_rows.append({
                'split': split_name, 'cell_type': ct,
                'jaccard_ot': round(jac_ot, 4), 'jaccard_ctrl': round(jac_ctrl, 4),
                'jaccard_gain': round(jac_ot - jac_ctrl, 4),
                'pathway_recall_ot': round(recall_ot, 4),
                'pathway_recall_ctrl': round(recall_ctrl, 4),
                'pathway_recall_gain': round(recall_ot - recall_ctrl, 4),
                'n_true_pathways': len(enr_true),
                'n_cells': int(ct_mask.sum())
            })

    pathway_df = pd.DataFrame(pathway_rows)
    pathway_df.to_csv(f'{cfg.results_dir}/pathway_recovery_nb13.csv', index=False)
    print('\nSaved: pathway_recovery_nb13.csv')
    print(pathway_df.to_string(index=False))

else:
    print('Pathway recovery skipped (gseapy or ctrl_means_gene not available).')
    pathway_df = pd.DataFrame()

gseapy available.
  [test] A549: DEG Jaccard  OT=0.493  ctrl=0.361  Δ=+0.132
  Enrichr error: Invalid organism 'Human'. Valid options are: c. elegans, caenorhabditis elegans, celegans, d. melanogaster, d. rerio, danio rerio, drosophila, drosophila melanogaster, enrichr, fish, fly, h. sapiens, homo sapiens, hs, hsapiens, human, m. musculus, mm, mouse, mus musculus, nematode, s. cerevisiae, saccharomyces, saccharomyces cerevisiae, worm, yeast, zebrafish
  Enrichr error: Invalid organism 'Human'. Valid options are: c. elegans, caenorhabditis elegans, celegans, d. melanogaster, d. rerio, danio rerio, drosophila, drosophila melanogaster, enrichr, fish, fly, h. sapiens, homo sapiens, hs, hsapiens, human, m. musculus, mm, mouse, mus musculus, nematode, s. cerevisiae, saccharomyces, saccharomyces cerevisiae, worm, yeast, zebrafish
  Enrichr error: Invalid organism 'Human'. Valid options are: c. elegans, caenorhabditis elegans, celegans, d. melanogaster, d. rerio, danio rerio, drosophila, droso

## 7) JOB C — Per-supergroup OOD analysis

For each supergroup: OT gain vs routing distance (how far were OOD drugs from the supergroup center).
This explains mechanistically why OOD gain is smaller than test gain.

In [11]:
# Use seed=42 predictions
all_drug_sg_42 = seed42['_all_drug_sg']

# Tanimoto similarity (RDKit) between OOD drug scaffold and its assigned supergroup
try:
    from rdkit import Chem
    from rdkit.Chem import DataStructs
    from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect
    RDKIT_OK = True
except ImportError:
    RDKIT_OK = False

def tanimoto(smi1, smi2):
    if not RDKIT_OK or smi1 is None or smi2 is None:
        return np.nan
    try:
        m1 = Chem.MolFromSmiles(str(smi1))
        m2 = Chem.MolFromSmiles(str(smi2))
        if m1 is None or m2 is None:
            return np.nan
        fp1 = GetMorganFingerprintAsBitVect(m1, 2, 2048)
        fp2 = GetMorganFingerprintAsBitVect(m2, 2, 2048)
        return float(DataStructs.TanimotoSimilarity(fp1, fp2))
    except Exception:
        return np.nan

# Load OOD routing table from NB12 if available, else rebuild
routing_path = f'{cfg.results_dir}/../results_nb12_ot_pilot/artifacts/ood_test_drug_routing.csv'
if os.path.exists(routing_path):
    routing_df = pd.read_csv(routing_path)
else:
    # Build from current seed=42 run
    routing_records = []
    for drug in set(cond_vals[is_ood]):
        sg_id = all_drug_sg_42.get(drug, -1)
        routing_records.append({'drug': drug,
                                 'scaffold': drug_to_scaffold.get(drug, 'unknown'),
                                 'split': 'ood',
                                 'supergroup': sg_id})
    for drug in set(cond_vals[is_test]):
        sg_id = all_drug_sg_42.get(drug, -1)
        routing_records.append({'drug': drug,
                                 'scaffold': drug_to_scaffold.get(drug, 'unknown'),
                                 'split': 'test',
                                 'supergroup': sg_id})
    routing_df = pd.DataFrame(routing_records)

# Per-supergroup gain (OOD)
sg_analysis_rows = []
pred_ot_ood   = seed42['_pred_ot_ood']
pred_ctrl_ood = seed42['_pred_ctrl_ood']
true_ood_arr  = seed42['_true_ood']
idxs_ood      = seed42['_idxs_ood']
drugs_ood     = cond_vals[idxs_ood]

n_sg = cfg.n_supergroups
for sg_id in range(n_sg):
    sg_drugs_ood = [d for d in set(drugs_ood) if all_drug_sg_42.get(d) == sg_id]
    if not sg_drugs_ood:
        continue
    sg_mask = np.isin(drugs_ood, sg_drugs_ood)
    if sg_mask.sum() == 0:
        continue

    m_ot   = compute_metrics(pred_ot_ood[sg_mask],   true_ood_arr[sg_mask], pred_ctrl_ood[sg_mask])
    m_ctrl = compute_metrics(pred_ctrl_ood[sg_mask], true_ood_arr[sg_mask], pred_ctrl_ood[sg_mask])
    gain   = m_ot['r2_top50'] - m_ctrl['r2_top50']

    # Mean routing distance from NB12 (proxy for OOD novelty)
    routing_sg = routing_df[(routing_df['split']=='ood') & (routing_df['supergroup']==sg_id)]
    mean_dist  = routing_sg['dist_to_sg_center'].mean() if ('dist_to_sg_center' in routing_sg.columns and len(routing_sg) > 0) else np.nan

    sg_analysis_rows.append({
        'supergroup': sg_id,
        'n_ood_drugs': len(sg_drugs_ood),
        'n_ood_cells': int(sg_mask.sum()),
        'ot_r2_top50': round(m_ot['r2_top50'], 4),
        'ctrl_r2_top50': round(m_ctrl['r2_top50'], 4),
        'gain_r2_top50': round(gain, 4),
        'mean_routing_dist': round(mean_dist, 4) if not np.isnan(mean_dist) else None,
    })

sg_df = pd.DataFrame(sg_analysis_rows).sort_values('gain_r2_top50', ascending=False)
sg_df.to_csv(f'{cfg.results_dir}/supergroup_ood_analysis_nb13.csv', index=False)

print('=' * 65)
print('PER-SUPERGROUP OOD ANALYSIS (seed=42)')
print('=' * 65)
print(sg_df.to_string(index=False))

# Correlation: gain vs routing distance
valid = sg_df.dropna(subset=['mean_routing_dist'])
if len(valid) >= 3:
    r, p = pearsonr(valid['mean_routing_dist'], valid['gain_r2_top50'])
    print(f'\nCorrelation (routing_dist vs OT gain): r={r:.3f}  p={p:.3f}')
    if r < -0.3:
        print('  → Larger routing distance = smaller OT gain: supports "OT works best for structurally similar drugs"')
    else:
        print('  → Weak correlation: routing distance alone does not explain OOD gain pattern.')

print('\nSaved: supergroup_ood_analysis_nb13.csv')

PER-SUPERGROUP OOD ANALYSIS (seed=42)
 supergroup  n_ood_drugs  n_ood_cells  ot_r2_top50  ctrl_r2_top50  gain_r2_top50  mean_routing_dist
          3            1         1696       0.4332         0.2559         0.1773             0.1873
          7            2         2595       0.5031         0.4477         0.0554             0.5724
          0           13        25265       0.5677         0.5663         0.0015             0.2368
          2            3         5911       0.5476         0.5461         0.0015             0.3743
          5            8        14399       0.5764         0.5762         0.0002             0.2549

Correlation (routing_dist vs OT gain): r=-0.229  p=0.711
  → Weak correlation: routing distance alone does not explain OOD gain pattern.

Saved: supergroup_ood_analysis_nb13.csv


## 8) Final summary — what to report in the paper

In [12]:
print('=' * 70)
print('NB13 FINAL SUMMARY — PAPER-READY NUMBERS')
print('=' * 70)

print('\n── Job A: Multi-seed CI ─────────────────────────────────────────')
key_metrics = ci_df[ci_df['metric'] == 'r2_top50'][['split','method','mean','ci_lo','ci_hi']]
print(key_metrics.to_string(index=False))

test_ot   = ci_df[(ci_df.split=='test') & (ci_df.method=='ot')   & (ci_df.metric=='r2_top50')].iloc[0]
test_ctrl = ci_df[(ci_df.split=='test') & (ci_df.method=='ctrl') & (ci_df.metric=='r2_top50')].iloc[0]
ood_ot    = ci_df[(ci_df.split=='ood')  & (ci_df.method=='ot')   & (ci_df.metric=='r2_top50')].iloc[0]
ood_ctrl  = ci_df[(ci_df.split=='ood')  & (ci_df.method=='ctrl') & (ci_df.metric=='r2_top50')].iloc[0]

print(f'\n  Test gain:  Δr²_top50 = {test_ot["mean"]-test_ctrl["mean"]:+.4f}')
print(f'  OOD  gain:  Δr²_top50 = {ood_ot["mean"]-ood_ctrl["mean"]:+.4f}')

ci_overlap_test = test_ot['ci_lo'] < test_ctrl['ci_hi']
ci_overlap_ood  = ood_ot['ci_lo']  < ood_ctrl['ci_hi']
print(f'  Test gain significant (non-overlapping CI): {not ci_overlap_test}')
print(f'  OOD  gain significant (non-overlapping CI): {not ci_overlap_ood}')

if not pathway_df.empty:
    print('\n── Job B: Pathway Recovery ──────────────────────────────────────')
    print(pathway_df[['split','cell_type','jaccard_ot','jaccard_ctrl',
                        'jaccard_gain','pathway_recall_ot','pathway_recall_ctrl']].to_string(index=False))

print('\n── Job C: Best/Worst Supergroups (OOD) ─────────────────────────')
print(sg_df[['supergroup','n_ood_drugs','gain_r2_top50','mean_routing_dist']].to_string(index=False))

print('\n── Recommendation ───────────────────────────────────────────────')
if not ci_overlap_test:
    print('  Test gain is SIGNIFICANT → use r²_top50 gain as primary metric in paper.')
else:
    print('  Test gain CIs OVERLAP → do NOT claim improvement; reframe as stability result.')

if not ci_overlap_ood:
    print('  OOD gain is SIGNIFICANT → OT generalizes to unseen scaffold drugs.')
else:
    print('  OOD gain CIs OVERLAP → honest framing: "OT maintains performance; does not degrade OOD"')
    print('  This is still publishable — many methods show OOD drop. Stability is a contribution.')

print('\nAll results saved to:', cfg.results_dir)

NB13 FINAL SUMMARY — PAPER-READY NUMBERS

── Job A: Multi-seed CI ─────────────────────────────────────────
split method   mean  ci_lo  ci_hi
 test     ot 0.5628 0.5627 0.5629
 test   ctrl 0.5291 0.5291 0.5291
  ood     ot 0.5600 0.5598 0.5603
  ood   ctrl 0.5500 0.5500 0.5500

  Test gain:  Δr²_top50 = +0.0337
  OOD  gain:  Δr²_top50 = +0.0100
  Test gain significant (non-overlapping CI): True
  OOD  gain significant (non-overlapping CI): True

── Job B: Pathway Recovery ──────────────────────────────────────
split cell_type  jaccard_ot  jaccard_ctrl  jaccard_gain  pathway_recall_ot  pathway_recall_ctrl
 test      A549      0.4925        0.3605        0.1320                0.0                  0.0
 test      K562      0.4286        0.2346        0.1940                0.0                  0.0
 test      MCF7      0.5748        0.2821        0.2928                0.0                  0.0
  ood      A549      0.4493        0.3514        0.0979                0.0                  0.0
  oo

## Appendix — chemCPA baseline scaffold

Run this cell only if you want to directly compare against chemCPA on your exact scaffold split.
Requires chemCPA to be installed. Uncomment and run separately.

In [13]:
# ── chemCPA comparison (optional, run separately) ─────────────────────────
# Uncomment to run. Requires: pip install chemCPA
# This gives the apples-to-apples comparison needed for the paper.
#
# !pip -q install chemCPA
# from chemCPA.model import ComPert
# ...
# See: https://github.com/theislab/chemCPA
#
# Minimum viable comparison:
# 1. Use your scaffold split (split_scaffold column from NB10)
# 2. Run chemCPA in 'unseen_drug' mode
# 3. Evaluate on same test/OOD splits with same r2_top50 metric
# 4. If your OT beats chemCPA → strong claim
#    If tied → 'comparable to SOTA with simpler method' → still publishable
print('chemCPA comparison cell — uncomment to run.')

chemCPA comparison cell — uncomment to run.
